Use the same observations for both fits. The generator draws a label with
probabilities $\boldsymbol{\pi}=(.40,.35,.25)^\top$, then a two-dimensional Gaussian
feature vector with the corresponding mean and common covariance

$$
\boldsymbol{\mu}_0=(-1,0)^\top,\quad \boldsymbol{\mu}_1=(1,.5)^\top,\quad
\boldsymbol{\mu}_2=(0,2)^\top,\qquad
\Sigma=\begin{pmatrix}1&.5\\.5&1\end{pmatrix}.
$$

The two original means and the covariance carry over from the binary project.
The starter supplies the generator, fitted Gaussian baseline, checker and
trainer. Use NumPy's `default_rng` with separate streams.

| Sample     |  Size |     Seed | Use                                       |
|:-----------|------:|---------:|:------------------------------------------|
| Training   |   300 | 20261116 | Fit both models and the feature scaling   |
| Validation |   180 | 20261117 | Select one penalty by log loss            |
| Test       | 2,000 | 20261118 | Evaluate both frozen fits once            |
| Checker    |     7 | 20261119 | Verify scores and derivatives             |

Standardize all splits with the training means and standard deviations
(`ddof=0`). Only the generator receives the population parameters.
Generate the test stream after model selection.

In [14]:
import numpy as np
from scipy.special import logsumexp

# ---- Population parameters (only the generator may see these) ----
PI = [0.40, 0.35, 0.25]
MU_OLD = [[-1.0, 0.0], [1.0, 0.5]]   # mu_0, mu_1 from the binary project
MU_NEW = [[0.0, 2.0]]                # mu_2
COV = np.array([[1.0, 0.5],
                [0.5, 1.0]])

SEEDS = {"train": 20261116, "valid": 20261117,
         "test": 20261118, "checker": 20261119}
SIZES = {"train": 300, "valid": 180, "test": 2000, "checker": 7}


def make_data(n, seed, pi=None, means=None, cov=COV):
    """Draw y ~ Categorical(pi), then X | y ~ N(mu_y, cov).

    Each call uses its own Generator, default_rng(seed), so every split is
    an independent stream: drawing one split never shifts another.
    Returns X of shape (n, 2) and integer y of shape (n,).
    """
    pi = np.asarray(PI if pi is None else pi, dtype=float)
    if means is None:
        means = np.vstack([MU_OLD, MU_NEW])
    means = np.asarray(means, dtype=float)
    if pi.ndim != 1 or len(pi) != len(means) or not np.isclose(pi.sum(), 1):
        raise ValueError("pi must be a probability vector with one entry per mean.")

    rng = np.random.default_rng(seed)
    y = rng.choice(len(pi), size=n, p=pi)
    z = rng.standard_normal((n, means.shape[1]))
    X = z @ np.linalg.cholesky(cov).T + means[y]
    return X, y


def fit_scaler(X_train):
    """Training-split mean and standard deviation (ddof=0)."""
    return X_train.mean(axis=0), X_train.std(axis=0, ddof=0)


def standardize(X, scaler):
    m, s = scaler
    return (X - m) / s


def make_splits(**kw):
    """Training, validation and checker splits, standardized with training stats.

    The test split is deliberately left out: call make_test(scaler)
    only after the penalty has been chosen on validation.
    """
    raw = {k: make_data(SIZES[k], SEEDS[k], **kw)
           for k in ("train", "valid", "checker")}
    scaler = fit_scaler(raw["train"][0])
    data = {k: (standardize(X, scaler), y) for k, (X, y) in raw.items()}
    return data, scaler


def make_test(scaler, **kw):
    """Generate and standardize the 2,000-point test split (once, at the end)."""
    X, y = make_data(SIZES["test"], SEEDS["test"], **kw)
    return standardize(X, scaler), y


if __name__ == "__main__":
    data, scaler = make_splits()
    for k, (X, y) in data.items():
        print(f"{k:8s} X{X.shape} y{y.shape} counts={np.bincount(y, minlength=3)}")
    print("train mean", data["train"][0].mean(0).round(12),
          "std", data["train"][0].std(0).round(12))

train    X(300, 2) y(300,) counts=[130  87  83]
valid    X(180, 2) y(180,) counts=[59 73 48]
checker  X(7, 2) y(7,) counts=[4 2 1]
train mean [-0.  0.] std [1. 1.]


No.6 Inspect `fit_gda` in the starter. On the seven-example checker sample, compare its normalized log scores to Gaussian log probabilities computed from the same estimated means, covariance and priors using a Cholesky solve. Record the largest absolute difference (target $< 10^{-12}$). Run the starter with `--gda` for this check alone.

Before training Softmax, predict whether the two fits must agree on a finite sample and whether one must win on every test set. Give a reason.

In [16]:
def fit_gda(X, y, eps=1e-3):
    # Fit from training samples only; return W (d,3), b (3,).
    n, d = X.shape
    counts = np.bincount(y, minlength=3)
    if len(counts) != 3 or np.any(counts == 0):
        raise ValueError("Require all three classes")
    means = np.stack([X[y == k].mean(0) for k in range(3)])
    R = X - means[y]
    S = R.T @ R/n + eps*np.eye(d)
    W = np.linalg.solve(S, means.T)
    b = np.log(counts/n) - .5*np.sum(means.T*W, axis=0)
    return W, b

def fit_gda_C(X, y, eps=1e-3):
    n, d = X.shape
    counts = np.bincount(y, minlength=3)
    if len(counts) != 3 or np.any(counts == 0):
        raise ValueError("Require all three classes")
    M = np.stack([X[y == k].mean(0) for k in range(3)])
    R = X - M[y]
    S = R.T @ R/n + eps*np.eye(d)
    # Using a Cholesky solve
    L = np.linalg.cholesky(S) # Σ = L @ L.T
    Y = np.linalg.solve(L, M.T) # L @ Y = M
    W = np.linalg.solve(L.T, Y) # L.T @ W = Y.
    b = np.log(counts/n) - .5*np.sum(M.T*W, axis=0)
    return W, b

#----------- Test --------------

def log_probs(X, W, b): # the scores 
    s = X @ W + b                      # s_ik = w_k^T x_i + b_k
    return s - logsumexp(s, axis=1, keepdims=True)

data, _ = make_splits()
Xtr, ytr = data["train"]
Xc, yc = data["checker"]

lp1 = log_probs(Xc, *fit_gda(Xtr, ytr))
lp2 = log_probs(Xc, *fit_gda_C(Xtr, ytr))
np.set_printoptions(precision=6, suppress=True)
print("fit_gda   log P(Y=k|x):\n", lp1)
print("fit_gda_C log P(Y=k|x):\n", lp2)
diff = np.max(np.abs(lp1 - lp2))
print(f"max |diff| = {diff:.3e}  (< 1e-12: {diff < 1e-12})")

fit_gda   log P(Y=k|x):
 [[ -0.214788  -2.414725  -2.264371]
 [ -0.037133  -6.36359   -3.360186]
 [ -0.241619  -1.558037  -5.49777 ]
 [ -0.003538 -10.920959  -5.651014]
 [ -0.109454  -2.712472  -3.28865 ]
 [ -4.571729  -0.032763  -3.821649]
 [ -0.581577  -1.980087  -1.194265]]
fit_gda_C log P(Y=k|x):
 [[ -0.214788  -2.414725  -2.264371]
 [ -0.037133  -6.36359   -3.360186]
 [ -0.241619  -1.558037  -5.49777 ]
 [ -0.003538 -10.920959  -5.651014]
 [ -0.109454  -2.712472  -3.28865 ]
 [ -4.571729  -0.032763  -3.821649]
 [ -0.581577  -1.980087  -1.194265]]
max |diff| = 8.882e-16  (< 1e-12: True)


No.7 *Implement and test.* Fill in `log_probs(Z)` and `loss_grad(X, y, W, b, lam)` in the starter code. `log_probs` returns an $(n, K)$ array. `loss_grad` returns $(J, \nabla_W J, \nabla_{\mathbf{b}} J)$.

Run the supplied checker with `--check`. It applies finite differences to every weight and every intercept with step $h = 10^{-6}$. Record the maximum relative error

$$
\max \frac{|g - g_{\text{num}}|}{\max(1, |g|, |g_{\text{num}}|)}
$$

The target is below $10^{-7}$.

Explain one check that could catch a bug even when your finite differences pass. Some options:

- row sums
- rowwise shifts
- extreme scores
- the intercept gradient when all scores are zero
- class-gradient sums at $\lambda = 0$

In [18]:
def log_prob(Z):
    n, K = Z.shape
    P = np.zeros((n, K))
    for i in range(n):
        mi = np.max(Z[i])
        for j in range(K):
            P[i][j] = Z[i][j] - mi - logsumexp(Z[i] - mi)
    return P

def loss_grad(X, y, W, b, lam):
    n = X.shape[0]
    d, K = W.shape
    J, dWJ, dbJ = 0, np.zeros((d, K)), np.zeros(n)
    Z = X @ W + b
    P = log_prob(Z)
    T = np.zeros((n, K))      
    for i in range(n):
        J += -P[i][y[i]]
        for j in range(K):
            P[i][j] = np.exp(P[i][j])
            if y[i] == j:
                T[i][j] = 1
    dWJ = (X.T @ (P - T))/n + lam * W
    dbJ = np.sum(P-T, axis=0)/n
    J = J/n + 0.5 * lam * np.sum(W ** 2)
    return J, dWJ, dbJ

#---------- Test ------------

data, scaler = make_splits()
X, y = data["checker"]

rng = np.random.default_rng(0)
W = rng.normal(size=(2, 3))
b = rng.normal(size=3)

lam = 0.1
h = 1e-6

J, dWJ, dbJ = loss_grad(X, y, W, b, lam)

dWJ_num = np.zeros_like(W)
dbJ_num = np.zeros_like(b)

# Numerical gradient with respect to W
for i in range(W.shape[0]):
    for k in range(W.shape[1]):
        W_plus = W.copy()
        W_minus = W.copy()

        W_plus[i, k] += h
        W_minus[i, k] -= h

        J_plus = loss_grad(X, y, W_plus, b, lam)[0]
        J_minus = loss_grad(X, y, W_minus, b, lam)[0]

        dWJ_num[i, k] = (J_plus - J_minus) / (2 * h)

# Numerical gradient with respect to b
for k in range(len(b)):
    b_plus = b.copy()
    b_minus = b.copy()

    b_plus[k] += h
    b_minus[k] -= h

    J_plus = loss_grad(X, y, W, b_plus, lam)[0]
    J_minus = loss_grad(X, y, W, b_minus, lam)[0]

    dbJ_num[k] = (J_plus - J_minus) / (2 * h)

# Normalized gradient errors
err_W = np.abs(dWJ - dWJ_num) / np.maximum(
    1, np.maximum(np.abs(dWJ), np.abs(dWJ_num))
)

err_b = np.abs(dbJ - dbJ_num) / np.maximum(
    1, np.maximum(np.abs(dbJ), np.abs(dbJ_num))
)

max_error = max(np.max(err_W), np.max(err_b))

print("Weight gradient error:", np.max(err_W))
print("Intercept gradient error:", np.max(err_b))
print("Maximum gradient error:", max_error)

assert max_error < 1e-7

Weight gradient error: 2.330184933896362e-10
Intercept gradient error: 1.0461645438830658e-10
Maximum gradient error: 2.330184933896362e-10


*Train.* For each $\lambda\in\{.001,.01,.1\}$ run the supplied 600
full-gradient updates. With $A=[X\ \boldsymbol{1}_n]$, the step size is $\eta=1/L$ and
$L=\|A\|_2^2/(2n)+\lambda$. You may use $\|C_i\|_2 \leqslant 1/2$ for each row Hessian
and the descent lemma. Explain the factor $\|A\|_2^2/n$ and why subtracting
the across-class mean of each weight row and of the intercepts preserves
descent. For each fit, record validation error, unpenalized validation log
loss and final gradient norm.

In [22]:
def full_gradD(X, y, W, b, lam):
    n = X.shape[0]
    d, K = W.shape
    Wf, bf = W, b
    A = np.hstack((X, np.ones((n, 1))))
    L = np.linalg.norm(A, ord=2)**2 / (2*n) + lam
    eta = 1 / L
    for i in range(600):
        J, dWJ, dbJ = loss_grad(X, y, Wf, bf, lam)
        Wf = Wf - eta * dWJ
        bf = bf - eta * dbJ
        # Parameter centering
        Wf = Wf - Wf.mean(axis=1, keepdims=True)
        bf = bf - bf.mean()
    return Wf, bf

def validError(X, y, W, b):
    n = X.shape[0]
    Z = X @ W + b
    e = 0
    for i in range(n):
        sort = np.argmax(Z[i]) # The softmax assign the sample to class k iff P(Y = k | X_i) is the max.
        if sort == y[i]:
            e += 1
    return e

def ulog_loss(X, y, W, b, lam):
    J = loss_grad(X, y, W, b, lam)[0]
    return J - 0.5 * lam * np.sum(W ** 2)

def gradNorm(X, y, W, b, lam):
    _, dWJ, dbJ = loss_grad(X, y, W, b, lam)
    return np.sqrt(np.linalg.norm(dWJ, ord=2)**2 + np.linalg.norm(dbJ, ord=2)**2)

#--------- Test ----------
data, scaler = make_splits()

X_train, y_train = data["train"]
X_valid, y_valid = data["valid"]

d = X_train.shape[1]
K = 3

W = np.zeros((d, K))
b = np.zeros(K)

results = []

for lam in [0.001, 0.01, 0.1]:

    Wf, bf = full_gradD(
        X_train, y_train, W, b, lam
    )

    error = validError(X_valid, y_valid, Wf, bf)

    logloss = ulog_loss(X_valid, y_valid, Wf, bf, lam)

    grad = gradNorm(X_train, y_train, Wf, bf, lam)

    results.append((lam, error, logloss, grad))

    print(f"Lambda: {lam}")
    print(f"Validation error: {error:.8f}")
    print(f"Validation log loss: {logloss:.8f}")
    print(f"Final gradient norm: {grad:.8e}")
    print()

Lambda: 0.001
Validation error: 138.00000000
Validation log loss: 0.60970748
Final gradient norm: 2.28949533e-10

Lambda: 0.01
Validation error: 137.00000000
Validation log loss: 0.61046245
Final gradient norm: 1.12062964e-16

Lambda: 0.1
Validation error: 129.00000000
Validation log loss: 0.69817314
Final gradient norm: 7.16428973e-17



Run the starter without flags for the fixed experiment.
Select the smallest validation log loss (exact ties go to larger $\lambda$).
Freeze both fits and compare them on the same test cases. Report each model's
error and mean log loss, the interval from 5(3), and the paired counts
$b$ (only Gaussian wrong) and $c$ (only Softmax wrong). Give the exact
two-sided paired error-test decision at $.05$ and compare the observed error
gap with your practical threshold. Revisit 6(6). Which results speak to
optimization and sampling variability? Use the generator assumptions to
discuss misspecification and irreducible overlap.

In [31]:
from scipy.stats import binomtest

threshold_pp = 100 * 0.03 # 3%

def evaluate(X, y, W, bias):
    log_p = log_probs(X, W, bias)  # Your three-argument function from Problem 6
    wrong = log_p.argmax(axis=1) != y
    mean_log_loss = -log_p[np.arange(len(y)), y].mean()
    return wrong, mean_log_loss

# 7(5): Train each candidate from zero and save its fitted parameters.
data, scaler = make_splits()
X_train, y_train = data["train"]
X_valid, y_valid = data["valid"]

fits = {}
val_loss = {}

print("lambda   validation error   validation log loss   final gradient norm")
for lam in (0.001, 0.01, 0.1):
    W0 = np.zeros((X_train.shape[1], 3))
    W, bias = full_gradD(X_train, y_train, W0, np.zeros(3), lam)
    fits[lam] = (W, bias)

    wrong, nll = evaluate(X_valid, y_valid, W, bias)
    val_loss[lam] = nll
    _, gW, gb = loss_grad(X_train, y_train, W, bias, lam)
    grad_norm = np.sqrt(np.sum(gW**2) + np.sum(gb**2))
    print(f"{lam:6g}   {wrong.mean():16.6f}   {nll:19.8f}   {grad_norm:.3e}")

# 7(6): Select the lowest validation log loss. Exact ties go to the larger lambda.
best = min(fits, key=lambda lam: (val_loss[lam], -lam))
Ws, bs = fits[best]
Wg, bg = fit_gda(X_train, y_train)

# Generate the test set once, after both fits are frozen.
X_test, y_test = make_test(scaler)
g_wrong, g_loss = evaluate(X_test, y_test, Wg, bg)
s_wrong, s_loss = evaluate(X_test, y_test, Ws, bs)
n = len(y_test)

# Problem 5(3): 95% Hoeffding interval for the selected Softmax model's error.
s_error = s_wrong.mean()
radius = threshold_pp
lo = max(0.0, s_error - radius)
hi = min(1.0, s_error + radius)

b = int(np.sum(g_wrong & ~s_wrong))  # Only Gaussian is wrong
c = int(np.sum(~g_wrong & s_wrong))  # Only Softmax is wrong
gap_pp = 100 * (c - b) / n          # Softmax error minus Gaussian error

print(f"\nSelected lambda = {best:g}")
print(f"Gaussian: test error = {g_wrong.mean():.6f}, mean log loss = {g_loss:.6f}")
print(f"Softmax:  test error = {s_error:.6f}, mean log loss = {s_loss:.6f}")
print(f"Softmax 95% error interval = [{lo:.6f}, {hi:.6f}]")
print(f"Only Gaussian wrong (b) = {b}; only Softmax wrong (c) = {c}")
print(f"Error gap (Softmax minus Gaussian) = {gap_pp:+.3f} percentage points")
comparison = "meets" if abs(gap_pp) >= threshold_pp else "falls below"
print(f"The absolute gap {comparison} the threshold of {threshold_pp:g} percentage points.")

p_value = binomtest(
    b, b + c, p=0.5,
    alternative="two-sided"
).pvalue

print(f"Exact paired-test p-value = {p_value:.8f}")
print("Reject H0" if p_value <= 0.05 else "Fail to reject H0")

lambda   validation error   validation log loss   final gradient norm
 0.001           0.233333            0.60970748   2.292e-10
  0.01           0.238889            0.61046245   1.282e-16
   0.1           0.283333            0.69817314   7.678e-17

Selected lambda = 0.001
Gaussian: test error = 0.223000, mean log loss = 0.538219
Softmax:  test error = 0.228000, mean log loss = 0.541029
Softmax 95% error interval = [0.000000, 1.000000]
Only Gaussian wrong (b) = 8; only Softmax wrong (c) = 18
Error gap (Softmax minus Gaussian) = +0.500 percentage points
The absolute gap falls below the threshold of 3 percentage points.
Exact paired-test p-value = 0.07551870
Fail to reject H0
